# Context-Aware Neural Recommendation Engine
## Lightweight Recommendation Demo

This notebook loads the saved Two-Tower embeddings and uses the project's existing recommendation service. It does not retrain the model.

## 1. Locate the project and saved model files

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'models' / 'two_tower').is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_DIR = ROOT / 'models' / 'two_tower'
required_files = [
    'user_embeddings.npy', 'user_ids.npy',
    'item_embeddings.npy', 'item_ids.npy',
]
missing = [name for name in required_files if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing model files: {missing}')

print('Project root:', ROOT)
print('Saved embedding files: all present')

## 2. Load and validate the saved embeddings

In [ ]:
user_ids = np.load(MODEL_DIR / 'user_ids.npy', allow_pickle=True)
item_ids = np.load(MODEL_DIR / 'item_ids.npy', allow_pickle=True)
user_embeddings = np.load(MODEL_DIR / 'user_embeddings.npy', mmap_mode='r')
item_embeddings = np.load(MODEL_DIR / 'item_embeddings.npy', mmap_mode='r')

assert user_embeddings.ndim == 2 and item_embeddings.ndim == 2
assert len(user_ids) == user_embeddings.shape[0]
assert len(item_ids) == item_embeddings.shape[0]
assert user_embeddings.shape[1] == item_embeddings.shape[1]

print('Users:', len(user_ids))
print('Items:', len(item_ids))
print('User embedding shape:', user_embeddings.shape)
print('Item embedding shape:', item_embeddings.shape)
print('Embedding dimensions:', user_embeddings.shape[1])

## 3. Generate recommendations using the existing service

In [ ]:
from src.recommendation.service import RecommendationService

# Select a customer known to exist in the saved user embeddings.
customer_id = str(user_ids[0])
k = 10

service = RecommendationService(model_dir=MODEL_DIR)
result = service.recommend(customer_id, k=k)
recommended_ids = [str(article_id) for article_id in result['recommendations']]

print('Customer ID:', customer_id)
print('Recommendations returned:', len(recommended_ids))
print('Retrieval service executed successfully:', len(recommended_ids) > 0)
recommended_ids

## 4. Display ranked recommendations and similarity scores

In [ ]:
# Calculate cosine similarities for this customer's saved embedding.
# The recommendation ranking itself comes from the existing service above.
user_vector = np.asarray(user_embeddings[0], dtype=np.float64)
item_matrix = np.asarray(item_embeddings, dtype=np.float64)
user_norm = np.linalg.norm(user_vector)
item_norms = np.linalg.norm(item_matrix, axis=1)
dot_products = item_matrix @ user_vector
denominators = item_norms * user_norm
all_scores = np.divide(
    dot_products,
    denominators,
    out=np.zeros_like(dot_products),
    where=denominators != 0,
)

score_by_id = {
    str(article_id): float(score)
    for article_id, score in zip(item_ids, all_scores)
}
results_df = pd.DataFrame({
    'Rank': range(1, len(recommended_ids) + 1),
    'Article ID': recommended_ids,
    'Cosine Similarity': [score_by_id.get(article_id, np.nan) for article_id in recommended_ids],
})
display(results_df)

print('Demo completed without retraining.')

## What this demonstrates
- The previously saved user and item embeddings can be loaded.
- The recommendation service returns ranked article IDs for a customer.
- Cosine similarity scores can be inspected for the returned articles.
- No model training or model fitting is performed in this notebook.